In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df             = pd.read_csv("eol_cleaned.csv",        low_memory=False)
outcome_matrix = pd.read_csv("eol_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("eol_cost_vector.csv",    index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("eol_unit_summary.csv")
cred_df        = pd.read_csv("eol_rq2_cred.csv")

print("All files loaded")
print(f"Cred steps: {len(cred_df)}")

In [ ]:
C_red       = cred_df['test_id'].tolist()
C_full      = outcome_matrix.columns.tolist()
C_full_cost = cost_vector['mean_cost_s'].sum()
C_red_cost  = sum(
    cost_vector.loc[s, 'mean_cost_s']
    for s in C_red if s in cost_vector.index
)

THRESHOLD       = 85
ESCAPE_PENALTY      = 10
PASS_RATE_WINDOW    = 50
PASS_RATE_THRESHOLD = 0.93

print(f"Full cost:           {C_full_cost:.2f} s")
print(f"Cred cost:           {C_red_cost:.2f} s")
print(f"Cred size:           {len(C_red)} / {len(C_full)} steps")
print(f"Max saving possible: {(C_full_cost-C_red_cost)/C_full_cost*100:.2f}%")

In [ ]:
all_failing = unit_summary[unit_summary['overall_outcome'] == 0]['unit_id'].tolist()

genuine_fail_units = []
abort_only_units   = []

for unit_id in all_failing:
    unit_steps    = df[df['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)

print(f"Genuine FAIL units:  {N_fail}")
print(f"ABORT-only excluded: {len(abort_only_units)}")

In [ ]:
seen, unit_stream = set(), []
for u in df['unit_id'].tolist():
    if u not in seen:
        unit_stream.append(u)
        seen.add(u)

unit_outcome_map = unit_summary.set_index('unit_id')['overall_outcome'].to_dict()
unit_pass_seq    = np.array([unit_outcome_map.get(u, 1) for u in unit_stream])

rolling_pass_rate = np.zeros(len(unit_stream))
for i in range(len(unit_stream)):
    window_start = max(0, i - PASS_RATE_WINDOW + 1)
    rolling_pass_rate[i] = unit_pass_seq[window_start:i+1].mean()

print(f"Units in stream:     {len(unit_stream)}")
print(f"Pass rate — mean:    {rolling_pass_rate.mean():.4f}")
print(f"Pass rate — min:     {rolling_pass_rate.min():.4f}")
print(f"% unstable:          {(rolling_pass_rate < PASS_RATE_THRESHOLD).mean()*100:.1f}%")

In [ ]:
def unit_is_defective(unit_id):
    return unit_id in failing_units

def cred_detects(unit_id):
    if unit_id not in outcome_matrix.index:
        return True
    row = outcome_matrix.loc[unit_id]
    for step in C_red:
        if step in row.index and row[step] == 0:
            return True
    return False

def get_reward(chosen_arm, unit_id):
    is_defective = unit_is_defective(unit_id)
    detected = escaped = False
    if chosen_arm == 0:
        reward = 0.0
        if is_defective:
            detected = True
    else:
        if not is_defective:
            reward = 1.0
        elif cred_detects(unit_id):
            reward, detected = 0.5, True
        else:
            reward, escaped = -ESCAPE_PENALTY, True
    return reward, detected, escaped

print("Helper functions defined")

In [ ]:
alpha_ts = [5.0, 1.0]
beta_ts  = [1.0, 1.0]

choices_ts, cost_ts, stability_log = [], [], []
rewards_log, escape_log = [], []
escape_ts = defects_ts = stability_overrides = 0

np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD
    stability_log.append(1 if stable else 0)

    theta_full = np.random.beta(alpha_ts[0], beta_ts[0])
    theta_red  = np.random.beta(alpha_ts[1], beta_ts[1])

    if not stable:
        theta_full += (PASS_RATE_THRESHOLD - pass_rate) * 10
        stability_overrides += 1

    chosen = 1 if theta_red > theta_full else 0
    choices_ts.append(chosen)

    reward, detected, escaped = get_reward(chosen, unit_id)
    cost_ts.append(C_full_cost if chosen == 0 else C_red_cost)

    if detected: defects_ts += 1
    if escaped:  escape_ts  += 1

    reward_clipped = max(0.0, min(1.0, reward))
    if reward_clipped >= 0.5:
        alpha_ts[chosen] += 1
    else:
        beta_ts[chosen]  += 1

    rewards_log.append(reward_clipped)
    escape_log.append(escape_ts)

print(f"Thompson Sampling complete")
print(f"Chose Cred:  {choices_ts.count(1)} ({choices_ts.count(1)/len(choices_ts)*100:.1f}%)")
print(f"Escaped:     {escape_ts}")

In [ ]:
counts_ucb, rewards_ucb_cum = [1, 1], [0.0, 0.0]
choices_ucb, cost_ucb = [], []
escape_ucb = defects_ucb = 0
np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD
    t         = i + 1
    ucb_full  = rewards_ucb_cum[0]/counts_ucb[0] + np.sqrt(2*np.log(t)/counts_ucb[0])
    ucb_red   = rewards_ucb_cum[1]/counts_ucb[1] + np.sqrt(2*np.log(t)/counts_ucb[1])
    if not stable:
        ucb_full += (PASS_RATE_THRESHOLD - pass_rate) * 10
    chosen = 1 if ucb_red > ucb_full else 0
    choices_ucb.append(chosen)
    reward, detected, escaped = get_reward(chosen, unit_id)
    cost_ucb.append(C_full_cost if chosen == 0 else C_red_cost)
    if detected: defects_ucb += 1
    if escaped:  escape_ucb  += 1
    reward_clipped = max(0.0, min(1.0, reward))
    rewards_ucb_cum[chosen] += reward_clipped
    counts_ucb[chosen]      += 1

print(f"UCB — Cred: {choices_ucb.count(1)/len(choices_ucb)*100:.1f}%, Escaped: {escape_ucb}")

In [ ]:
EPSILON = 0.1
mean_rewards_eg, counts_eg = [0.0, 0.0], [1, 1]
choices_eg, cost_eg = [], []
escape_eg = defects_eg = 0
np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD
    if np.random.random() < EPSILON:
        chosen = np.random.randint(0, 2)
    else:
        scores = mean_rewards_eg.copy()
        if not stable:
            scores[0] += (PASS_RATE_THRESHOLD - pass_rate) * 10
        chosen = int(np.argmax(scores))
    choices_eg.append(chosen)
    reward, detected, escaped = get_reward(chosen, unit_id)
    cost_eg.append(C_full_cost if chosen == 0 else C_red_cost)
    if detected: defects_eg += 1
    if escaped:  escape_eg  += 1
    reward_clipped = max(0.0, min(1.0, reward))
    mean_rewards_eg[chosen] = (
        mean_rewards_eg[chosen] * counts_eg[chosen] + reward_clipped
    ) / (counts_eg[chosen] + 1)
    counts_eg[chosen] += 1

print(f"EG — Cred: {choices_eg.count(1)/len(choices_eg)*100:.1f}%, Escaped: {escape_eg}")

In [ ]:
def compute_metrics(choices, cost_log, escaped, label):
    total    = len(choices)
    pct_cred = choices.count(1) / total * 100
    cost_mu  = np.mean(cost_log)
    saving   = (C_full_cost - cost_mu) / C_full_cost * 100
    esc_risk = escaped / max(N_fail, 1)
    return {
        'Algorithm':     label,
        'Cred (%)':      round(pct_cred, 1),
        'Saving (%)':    round(saving, 2),
        'Escaped':       escaped,
        'R_hat':         round(esc_risk, 6),
        'Quality OK':    'YES' if esc_risk*1e6 <= THRESHOLD else 'NO'
    }

static_escaped = sum(1 for u in failing_units if not cred_detects(u))
static_saving  = (C_full_cost - C_red_cost) / C_full_cost * 100

rows = [
    {'Algorithm':'Baseline (Cfull)', 'Cred (%)':0.0,   'Saving (%)':0.0,
     'Escaped':0, 'R_hat':0.0, 'Quality OK':'YES'},
    {'Algorithm':'Static Cred (RQ1)', 'Cred (%)':100.0, 'Saving (%)':round(static_saving,2),
     'Escaped':static_escaped, 'R_hat':round(static_escaped/N_fail,6), 'Quality OK':'YES' if static_escaped/N_fail*1e6 <= THRESHOLD else 'NO'},
    compute_metrics(choices_eg,  cost_eg,  escape_eg,  'Epsilon-Greedy'),
    compute_metrics(choices_ucb, cost_ucb, escape_ucb, 'UCB'),
    compute_metrics(choices_ts,  cost_ts,  escape_ts,  'Thompson Sampling'),
]

results_df = pd.DataFrame(rows)
results_df.to_csv("eol_results_comparison.csv", index=False)

print("\nEOL TRAINING RESULTS")
print("="*70)
print(results_df.to_string(index=False))

In [ ]:
def style_ax(ax):
    ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    ax.yaxis.set_major_locator(plt.MaxNLocator(10))
    ax.tick_params(axis='both', labelsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

exec_index   = np.arange(len(unit_stream))
window       = 300
choices_arr  = np.array(choices_ts)
rolling_cred = pd.Series(choices_arr).rolling(window=window, min_periods=1).mean().values

unstable_mask = rolling_pass_rate < PASS_RATE_THRESHOLD

fig, ax = plt.subplots(figsize=(9, 5))

# Shade instability regions
in_region, start_idx = False, None
for i, unstable in enumerate(unstable_mask):
    if unstable and not in_region:
        in_region, start_idx = True, i
    elif not unstable and in_region:
        ax.axvspan(start_idx, i, color='red', alpha=0.15, label='_nolegend_')
        in_region = False
if in_region:
    ax.axvspan(start_idx, len(unstable_mask)-1, color='red', alpha=0.15)

ax.plot(exec_index, rolling_cred, color='black', linewidth=2,
        label=f'Thompson Sampling (w={window})')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')

# Add instability legend patch
from matplotlib.patches import Patch
handles, labels = ax.get_legend_handles_labels()
handles.append(Patch(color='red', alpha=0.15, label='Unstable region'))
ax.legend(handles=handles, fontsize=10)

ax.set_xlabel('Execution index (EOL events in time order)', fontsize=14, fontweight='bold')
ax.set_ylabel('Fraction selecting Cred', fontsize=14, fontweight='bold')

# ── Replace style_ax and ylim with correct limits ────────────
ax.set_ylim(0, 1)
ax.set_xlim(0, 9500)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.xaxis.set_major_locator(plt.MultipleLocator(600))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig('eol_fig4_arm_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eol_fig4_arm_selection.png")

In [ ]:
mab_log_df = pd.DataFrame({
    'unit_id':           unit_stream,
    'chosen_arm_ts':     choices_ts,
    'chosen_arm_ucb':    choices_ucb,
    'chosen_arm_eg':     choices_eg,
    'reward_ts':         rewards_log,
    'cost_ts':           cost_ts,
    'process_stable':    stability_log,
    'rolling_pass_rate': rolling_pass_rate.tolist(),
})
mab_log_df.to_csv("eol_mab_log.csv", index=False)

print("Saved: eol_mab_log.csv, eol_results_comparison.csv")
print("\nTraining complete — paste results table and arm selection figure, then we do validation.")